In [1]:
import os
import argparse
import h5py
os.environ['HF_HOME'] = '/om/user/ericjm/.cache/huggingface'

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset


In [2]:
dataset = load_dataset("skylion007/openwebtext", split='train', streaming=True)

/om2/user/ericjm/miniconda3/envs/features/lib/python3.11/site-packages/datasets/load.py:1454: FutureWarning: The repository for skylion007/openwebtext contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/skylion007/openwebtext
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


# Testing LowRankAutoEncoderTopK

In [1]:
from structured_sae.ops import low_rank_mmm
from structured_sae.trainers.low_rank_topk import LowRankAutoEncoderTopK
from structured_sae.dictionary_learning.trainers.top_k import AutoEncoderTopK

In [2]:
ae = AutoEncoderTopK(512, 8000, k=32)
ae_s = LowRankAutoEncoderTopK(512, 8000, k=32, rank=100)

In [3]:
ae.encoder.weight.var().item()

0.0006508661317639053

In [4]:
low_rank_mmm(ae_s.enc_W1, ae_s.enc_W2).var().item()

0.000653135939501226

In [5]:
ae.decoder.weight.var().item()

0.0019531246507540345

In [6]:
low_rank_mmm(ae_s.dec_W1, ae_s.dec_W2).var().item()

0.0019594079349189997

In [7]:
ae.decoder.weight.norm(dim=0)

tensor([1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<LinalgVectorNormBackward0>)

In [8]:
low_rank_mmm(ae_s.dec_W1, ae_s.dec_W2).norm(dim=0)

tensor([0.9931, 0.9514, 1.0055,  ..., 1.0647, 0.9917, 0.9653],
       grad_fn=<LinalgVectorNormBackward0>)

In [10]:
ae_s.decoder_feature(0).norm()

tensor(0.9931, grad_fn=<LinalgVectorNormBackward0>)

In [11]:
# check alignment between encoder and decoder features
e0 = ae_s.encoder_feature(0)
d0 = ae_s.decoder_feature(0)
# compute cosine sim
(e0 @ d0) / (e0.norm() * d0.norm())

tensor(1.0000, grad_fn=<DivBackward0>)

In [12]:
e1 = ae_s.encoder_feature(1)
d1 = ae_s.decoder_feature(1)
(e1 @ d1) / (e1.norm() * d1.norm())

tensor(1.0000, grad_fn=<DivBackward0>)

# Testing SumKroneckerAutoEncoderTopK

In [11]:
from structured_sae.ops import sum_kronecker_mmm
from structured_sae.trainers.sum_kronecker_topk import SumKroneckerAutoEncoderTopK
from structured_sae.dictionary_learning.trainers.top_k import AutoEncoderTopK

In [12]:
ae = AutoEncoderTopK(512, 8000, k=32)
ae_s = SumKroneckerAutoEncoderTopK(512, 8000, k=32, r=16, d1=100, d2=32, d3=80, d4=16)

In [13]:
ae.encoder.weight.var().item()

0.0006505738128907979

In [14]:
(sum_kronecker_mmm(ae_s.enc_L, ae_s.enc_R) @ ae_s.enc_V).var().item()

0.0006479373550973833

In [15]:
ae.decoder.weight.var().item()

0.0019531246507540345

In [16]:
(ae_s.dec_V @ sum_kronecker_mmm(ae_s.dec_L, ae_s.dec_R)).var().item()

0.001943812589161098

In [17]:
(ae_s.dec_V @ sum_kronecker_mmm(ae_s.dec_L, ae_s.dec_R)).norm(dim=0)

tensor([0.9582, 1.0005, 0.9809,  ..., 0.9722, 0.9562, 0.9070],
       grad_fn=<LinalgVectorNormBackward0>)

In [18]:
ae_s.decoder_feature(0).norm()

tensor(0.9582, grad_fn=<LinalgVectorNormBackward0>)

In [19]:
# inner product now
e0 = ae_s.encoder_feature(0)
d0 = ae_s.decoder_feature(0)
# compute cosine sim
(e0 @ d0) / (e0.norm() * d0.norm())

tensor(1., grad_fn=<DivBackward0>)

In [20]:
e1 = ae_s.encoder_feature(1)
d1 = ae_s.decoder_feature(1)
(e1 @ d1) / (e1.norm() * d1.norm())

tensor(1.0000, grad_fn=<DivBackward0>)

# Testing BlockDiagonalAutoEncoderTopK

In [2]:
from structured_sae.ops import block_diagonal_mmm
from structured_sae.trainers.block_diagonal_topk import BlockDiagonalAutoEncoderTopK
from structured_sae.dictionary_learning.trainers.top_k import AutoEncoderTopK

In [3]:
ae = AutoEncoderTopK(512, 8000, k=32)
ae_s = BlockDiagonalAutoEncoderTopK(512, 8000, k=32, blocks=8, proj_dim=768*8)

In [4]:
ae.encoder.weight.var().item()

0.0006510515231639147

In [5]:
(block_diagonal_mmm(ae_s.enc_B) @ ae_s.enc_V).var().item()

0.0006513851112686098

In [6]:
ae.decoder.weight.var().item()

0.0019531245343387127

In [7]:
(ae_s.dec_V @ block_diagonal_mmm(ae_s.dec_B)).var().item()

0.001954154809936881

In [8]:
(ae_s.dec_V @ block_diagonal_mmm(ae_s.dec_B)).norm(dim=0)

tensor([0.9571, 1.0036, 1.0060,  ..., 0.9939, 0.9868, 1.0182],
       grad_fn=<LinalgVectorNormBackward0>)

In [9]:
ae_s.decoder_feature(0).norm()

tensor(0.9571, grad_fn=<LinalgVectorNormBackward0>)

In [10]:
# inner product now
e0 = ae_s.encoder_feature(0)
d0 = ae_s.decoder_feature(0)
# compute cosine sim
(e0 @ d0) / (e0.norm() * d0.norm())

tensor(1.0000, grad_fn=<DivBackward0>)

In [11]:
e1 = ae_s.encoder_feature(1)
d1 = ae_s.decoder_feature(1)
(e1 @ d1) / (e1.norm() * d1.norm())

tensor(1., grad_fn=<DivBackward0>)

# Testing HDF5ActivationBuffer

In [9]:
from structured_sae.utils import HDF5ActivationBuffer
from tqdm.auto import tqdm

In [2]:
def setup_notebook():
    try:
        from IPython import get_ipython

        ipython = get_ipython()
        ipython.magic("load_ext autoreload")
        ipython.magic("autoreload 2")

    except:
        pass

setup_notebook()

/tmp/ipykernel_4161886/264644983.py:6: DeprecationWarning: `magic(...)` is deprecated since IPython 0.13 (warning added in 8.1), use run_line_magic(magic_name, parameter_s).
  ipython.magic("load_ext autoreload")
/tmp/ipykernel_4161886/264644983.py:7: DeprecationWarning: `magic(...)` is deprecated since IPython 0.13 (warning added in 8.1), use run_line_magic(magic_name, parameter_s).
  ipython.magic("autoreload 2")


In [3]:
!nvidia-smi

Tue Aug 13 18:27:00 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.14              Driver Version: 550.54.14      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:C4:00.0 Off |                    0 |
| N/A   37C    P0             54W /  300W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [14]:
buffer = HDF5ActivationBuffer('../activations/gpt2-layer7.h5', device='cuda:0', buffer_size=2**17)

skip: 16
buffer shape: torch.Size([131072, 768])


In [15]:
batch = next(buffer)

batch_idxs: tensor([     0,     16,     32,  ..., 131024, 131040, 131056])
buffer idx after increment: 1


In [16]:
for _ in tqdm(range(1000)):
    batch = next(buffer)

  0%|          | 0/1000 [00:00<?, ?it/s]

batch_idxs: tensor([     1,     17,     33,  ..., 131025, 131041, 131057])
buffer idx after increment: 2
batch_idxs: tensor([     2,     18,     34,  ..., 131026, 131042, 131058])
buffer idx after increment: 3
batch_idxs: tensor([     3,     19,     35,  ..., 131027, 131043, 131059])
buffer idx after increment: 4
batch_idxs: tensor([     4,     20,     36,  ..., 131028, 131044, 131060])
buffer idx after increment: 5
batch_idxs: tensor([     5,     21,     37,  ..., 131029, 131045, 131061])
buffer idx after increment: 6
batch_idxs: tensor([     6,     22,     38,  ..., 131030, 131046, 131062])
buffer idx after increment: 7
batch_idxs: tensor([     7,     23,     39,  ..., 131031, 131047, 131063])
buffer idx after increment: 8
batch_idxs: tensor([     8,     24,     40,  ..., 131032, 131048, 131064])
buffer idx after increment: 9
batch_idxs: tensor([     9,     25,     41,  ..., 131033, 131049, 131065])
buffer idx after increment: 10
batch_idxs: tensor([    10,     26,     42,  ..., 1310

In [11]:
batch

tensor([[-3.1939,  1.0275, -0.9501,  ...,  2.0706, -0.1894,  0.1624],
        [-1.1012,  2.0268, -2.1118,  ..., -0.8310,  1.3228,  1.1416],
        [ 0.3104,  0.4586,  0.0908,  ...,  1.1707,  1.6622, -1.8666],
        ...,
        [ 0.2713,  0.0328, -1.4632,  ...,  0.6282, -0.4957, -2.9572],
        [ 0.3771,  1.1603,  1.9759,  ...,  3.6561,  0.1155,  1.1041],
        [-3.5252, -2.5160,  3.0126,  ...,  1.8003, -2.1378,  0.6356]],
       device='cuda:0')

In [13]:
2**13 * 1_000

8192000